### Getting Started 

#### Install Google Gen AI SDK for Python

In [2]:
%pip install --upgrade --quiet google-genai

Note: you may need to restart the kernel to use updated packages.


#### Restart runtime
To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

In [3]:
# restart the kernel after libraries are loaded
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

/var/tmp/ipykernel_6158/952386123.py:4: DeprecationWarning: `IPython.Application` is only a re-export of `traitlets.config.application.Application`; import it from traitlets directly. Accessing it here triggers an import of `IPython.core.application`, which is no longer imported when IPython is -- import that module explicitly if you rely on that import happening, in particular if you also rely on other submodules being transitively imported as a side effect.
  app = IPython.Application.instance()


{'status': 'ok', 'restart': True}

### Set Google Cloud project information and create client

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).



In [1]:
# Define project information
PROJECT_ID = "qwiklabs-gcp-04-683f270f4931"  # @param {type:"string"}
LOCATION = "global"

# Create the API client
from google import genai
client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

#### Import libraries


In [2]:
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    Tool,
    Part
)

### Task 3. Create a function call using Gemini

In [3]:
# Task 3.1
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling
# Load Gemini 3.5 Flash Model
model_id = "gemini-3.5-flash"

In [5]:
# Task 3.2
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling

get_current_weather_func = FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather in a given location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "Location"
            }
        }
    },
)

In [7]:
# Task 3.3
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling

weather_tool = Tool(
    function_declarations=[get_current_weather_func],
)

In [10]:
# Task 3.4
# use the following documentation to assist you complete this cell
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/function-calling

prompt = "What is the weather like in Boston?"

response = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=GenerateContentConfig(
        tools=[weather_tool],
        temperature=0,
    ),
)

response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              id=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b'\x01\x8f=k_\xfe\x00\x83\x03m\x88\xf1X\xf8\xe3y\xf3_V\x8b\xb9\xa8\x1d\x80\xda\xebe7\\@\xac\x9d\xc1\x12\xc5\xf2bb4\xb1n\xab\xc7\x98\x1es\xa3\xd4r\x942\xd2\x90\xc4%W\x16u\xbd\xe9~\xf8\xa8vT\xea\xf5\x87\xe8\x99\xf0\r\x01\x9e\xdaB=g!U\x83\xe2\xf5\xa7+l\xe0\xbauS\xaa\x13:\xa6 \x11\xe5...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>
    ),
  ],
  create_time=datetime.datetime(2026, 10, 23, 5, 12, 43, 938014, tzinfo=TzInfo(0)),
  model_version='gemini-3.5-flash',
  response_id='y-zaap6gOePI4PoPoJKuAQ',
  sdk_http_response=HttpResponse(
    headers=<dict len=9>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candi

### Task 4. Describe video contents using Gemini

In [11]:
# Run the following cell to import required libraries 
from google.genai.types import (
    GenerationConfig,
    Image,
    Part,
)

In [16]:
# Task 4.1
# Load the correct Gemini model use the following documentation to assist:
# https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/overview#supported-use-cases
# Load Gemini 3.5 Flash Model
multimodal_model = "gemini-3.5-flash"

In [17]:
import http.client
import typing
import urllib.request

import IPython.display
from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps


def display_images(
    images: typing.Iterable[Image],
    max_width: int = 600,
    max_height: int = 350,
) -> None:
    for image in images:
        pil_image = typing.cast(PIL_Image.Image, image._pil_image)
        if pil_image.mode != "RGB":
            # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
            pil_image = pil_image.convert("RGB")
        image_width, image_height = pil_image.size
        if max_width < image_width or max_height < image_height:
            # Resize to display a smaller notebook image
            pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
        IPython.display.display(pil_image)


def get_image_bytes_from_url(image_url: str) -> bytes:
    with urllib.request.urlopen(image_url) as response:
        response = typing.cast(http.client.HTTPResponse, response)
        image_bytes = response.read()
    return image_bytes


def load_image_from_url(image_url: str) -> Image:
    image_bytes = get_image_bytes_from_url(image_url)
    return Image.from_bytes(image_bytes)


def display_content_as_image(content: str | Image | Part) -> bool:
    if not isinstance(content, Image):
        return False
    display_images([content])
    return True


def display_content_as_video(content: str | Image | Part) -> bool:
    if not isinstance(content, Part):
        return False
    part = typing.cast(Part, content)
    file_path = part.file_data.file_uri.removeprefix("gs://")
    video_url = f"https://storage.googleapis.com/{file_path}"
    IPython.display.display(IPython.display.Video(video_url, width=600))
    return True


def print_multimodal_prompt(contents: list[str | Image | Part]):
    """
    Given contents that would be sent to Gemini,
    output the full multimodal prompt for ease of readability.
    """
    for content in contents:
        if display_content_as_image(content):
            continue
        if display_content_as_video(content):
            continue
        print(content)

In [18]:
# Task 4.2 Generate a video description
# https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/sdk-for-gemini/gemini-sdk-overview-reference#generate-content-from-video
# https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference#sample-requests-text-stream-response

prompt = """
What is shown in this video?
Where should I go to see it?
What are the top 5 places in the world that look like this?
"""

video = Part.from_uri(
    file_uri="gs://github-repo/img/gemini/multimodality_usecases_overview/mediterraneansea.mp4",
    mime_type="video/mp4",
)

contents = [prompt, video]

responses = client.models.generate_content_stream(
    model=multimodal_model,
    contents=contents
)

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text or "", end="")

-------Prompt--------

What is shown in this video?
Where should I go to see it?
What are the top 5 places in the world that look like this?




-------Response--------
The location shown in this video is the historic **Kaleiçi Old Harbor (Yat Limanı)** in **Antalya, Turkey**. 

The video captures the dramatic limestone cliffs, the turquoise waters of the Mediterranean Sea, the ancient breakwater with its lighthouse, and the modern city of Antalya perched directly atop the cliffs.

---

### Top 5 Places in the World That Look Like This

If you love the combination of dramatic coastal cliffs, historic harbors, and cities built right on the edge of the sea, here are five spectacular destinations around the world that offer a similar look and feel:

#### 1. Bonifacio, Corsica (France)
* **Why it looks similar:** Bonifacio is famous for its medieval old town, which sits precariously on the edge of sheer white limestone cliffs rising 70 meters (230 feet) above the Mediterranean Sea. Like Antalya, it features a natural harbor sheltered below the towering cliffs.

#### 2. Polignano a Mare, Puglia (Italy)
* **Why it looks similar:** T